# YOLO 워크숍 시작 노트북 — 우리 팀 프로젝트

이 노트북은 워크숍 프로젝트의 **뼈대**입니다. 주제와 데이터만 정하면, 데이터 받기 → 형식 맞추기 → 학습 → 평가 → 개선 실험 → 결과 정리까지 같은 순서로 진행할 수 있습니다.
`✏️` 표시가 있는 셀은 우리 팀 내용으로 **직접 채우는** 곳입니다.

| 단계 | 할 일 | 체크포인트 |
|---|---|---|
| 1 | 프로젝트 카드 작성 | CP1 주제·데이터 확정 |
| 2 | 데이터 받기 (캐글 / URL / Drive / 예시 데이터) | CP1 |
| 3 | 형식 확인·변환·분할 → `data.yaml` | CP1 |
| 4 | 데이터 살펴보기 | CP2 |
| 5 | 베이스라인 학습 | CP2 첫 학습 |
| 6 | 평가 — 지표, 임계값, 오류 분석 | CP3 |
| 7 | 개선 실험 기록 | CP3 개선 |
| 8 | 데모 — 새 이미지·영상에 적용 | CP4 |
| 9 | 결과 정리 — 요약 리포트 자동 생성 | CP4 결과 공유 |

**실행 환경**: Google Colab, 런타임 유형 **T4 GPU**. 결과는 Google Drive `MyDrive/yolo_workshop/<프로젝트 이름>`에 저장되므로 다음 시간에 이어서 할 수 있습니다.

---
## 1. 프로젝트 카드 ✏️

아래 셀의 값을 우리 팀 내용으로 바꾸고 실행합니다. 이 내용은 마지막 결과 리포트에 그대로 들어갑니다.
- `PROJECT_NAME`은 **영문·숫자·밑줄**만 씁니다 (폴더 이름으로 쓰임).
- `SUCCESS`에는 '무엇을 얼마나 잘하면 성공인지'를 숫자로 적습니다. 예: `안전모 미착용을 테스트 이미지에서 재현율 0.8 이상으로 찾는다`

In [ ]:
PROJECT_NAME = 'ppe_helmet_check'          # ✏️ 영문·숫자·밑줄
TEAM = '홍길동, 김철수'                     # ✏️
PROBLEM = '공사 현장 사진에서 안전모를 쓰지 않은 작업자를 찾는다'          # ✏️ 누가 어떤 상황에서 무엇을 알고 싶은가
SUCCESS = '테스트 세트에서 no_helmet 클래스 재현율 0.8 이상'            # ✏️ 숫자로
TASK = 'detect'                             # 'detect'(박스) 또는 'segment'(마스크)

---
## 준비 — 설치와 저장 위치

In [ ]:
!pip -q install ultralytics kagglehub tabulate

In [ ]:
import os, re, json, glob, time, shutil, random, zipfile
from pathlib import Path
from collections import Counter, defaultdict
import xml.etree.ElementTree as ET

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import Image, display
from ultralytics import YOLO

print('GPU 사용 가능:', torch.cuda.is_available())

try:
    from google.colab import drive, files, userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

assert re.fullmatch(r'[A-Za-z0-9_]+', PROJECT_NAME), 'PROJECT_NAME은 영문·숫자·밑줄만 쓰세요'
if IN_COLAB:
    drive.mount('/content/drive')
    SAVE_DIR = Path('/content/drive/MyDrive/yolo_workshop') / PROJECT_NAME
else:
    SAVE_DIR = Path('outputs') / PROJECT_NAME
SAVE_DIR.mkdir(parents=True, exist_ok=True)
RAW = Path('raw_data')           # 받은 원본 (런타임 로컬 디스크)
DATA = Path('dataset')           # YOLO 형식으로 정리한 데이터
print('결과 저장 위치:', SAVE_DIR.resolve())

def save_out(path):
    dst = SAVE_DIR / Path(path).name
    if Path(path).resolve() != dst.resolve():
        shutil.copy(path, dst)
    return dst

def show_grid(imgs, titles=None, cols=3, size=4):
    rows = (len(imgs) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(size * cols, size * rows))
    for ax, i in zip(np.array(axes).ravel(), range(rows * cols)):
        ax.axis('off')
        if i < len(imgs):
            ax.imshow(imgs[i][..., ::-1])
            if titles:
                ax.set_title(titles[i], fontsize=9)
    plt.tight_layout(); plt.show()

---
## 2. 데이터 받기 ✏️

`DATA_SOURCE`를 하나 고르고, 그 아래 해당 값만 채웁니다.

| `DATA_SOURCE` | 언제 | 채울 값 |
|---|---|---|
| `'kaggle'` | 캐글 공개 데이터셋 | `KAGGLE_SLUG` — 데이터셋 주소 `kaggle.com/datasets/` 뒤의 `작성자/이름` |
| `'url'` | 공공데이터·GitHub 등에서 **zip 직접 다운로드 링크**가 있을 때 | `ZIP_URL` |
| `'drive'` | AI Hub처럼 따로 받아 **내 Drive에 올려 둔 zip** | `DRIVE_ZIP` |
| `'ultralytics'` | 연습용 예시 데이터 (설정 이름만으로 자동 다운로드) | `ULTRA_YAML` (예: `'construction-ppe.yaml'`) |

**캐글 인증** — 공개 데이터셋은 로그인 없이 받아지는 경우가 많습니다. 다운로드 오류(401/403)가 나면 인증을 설정합니다.
1. 캐글 웹사이트 → 오른쪽 위 프로필 → **Settings → API**에서 토큰을 만듭니다.
2. Colab 왼쪽의 **열쇠 아이콘(보안 비밀)** 에 `KAGGLE_API_TOKEN`이라는 이름으로 토큰 값을 등록하고 '노트북 액세스'를 켭니다.
   예전 방식의 `kaggle.json`(username, key)을 받았다면 `KAGGLE_USERNAME`, `KAGGLE_KEY` 두 개로 등록합니다.
3. 코드에 키를 직접 적지 마세요. 노트북을 공유하면 키도 함께 퍼집니다.

In [ ]:
DATA_SOURCE = 'ultralytics'                       # ✏️ 'kaggle' | 'url' | 'drive' | 'ultralytics'
KAGGLE_SLUG = 'snehilsanyal/construction-site-safety-image-dataset-roboflow'   # ✏️ DATA_SOURCE='kaggle'일 때
ZIP_URL = ''                                      # ✏️ DATA_SOURCE='url'일 때
DRIVE_ZIP = '/content/drive/MyDrive/my_data.zip'  # ✏️ DATA_SOURCE='drive'일 때
ULTRA_YAML = 'construction-ppe.yaml'              # ✏️ DATA_SOURCE='ultralytics'일 때

In [ ]:
def unzip_all(src_dir):
    """폴더 안의 zip을 모두 풀기 (압축 안에 압축이 있는 경우 대비)"""
    for z in list(Path(src_dir).rglob('*.zip')):
        with zipfile.ZipFile(z) as f:
            f.extractall(z.parent / z.stem)
        z.unlink()

RAW.mkdir(exist_ok=True)
ULTRA_DATA = None
if DATA_SOURCE == 'kaggle':
    if IN_COLAB:                       # 보안 비밀에 등록된 값이 있으면 사용
        for key in ['KAGGLE_API_TOKEN', 'KAGGLE_USERNAME', 'KAGGLE_KEY']:
            try:
                os.environ[key] = userdata.get(key)
            except Exception:
                pass
    import kagglehub
    path = kagglehub.dataset_download(KAGGLE_SLUG)
    shutil.copytree(path, RAW / 'kaggle', dirs_exist_ok=True)
elif DATA_SOURCE == 'url':
    assert ZIP_URL, 'ZIP_URL을 채우세요'
    !wget -q -O raw_data/download.zip "{ZIP_URL}"
elif DATA_SOURCE == 'drive':
    shutil.copy(DRIVE_ZIP, RAW / Path(DRIVE_ZIP).name)
elif DATA_SOURCE == 'ultralytics':
    from ultralytics.data.utils import check_det_dataset
    ULTRA_DATA = check_det_dataset(ULTRA_YAML)
    shutil.copytree(ULTRA_DATA['path'], RAW / 'ultralytics', dirs_exist_ok=True)
else:
    raise ValueError(DATA_SOURCE)
unzip_all(RAW)

IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
ext_count = Counter(p.suffix.lower() for p in RAW.rglob('*') if p.is_file())
print('받은 파일 종류:', dict(ext_count.most_common(8)))

In [ ]:
# 폴더 구조 훑어보기 (깊이 3까지, 폴더마다 파일 수)
def tree(root, depth=3):
    root = Path(root)
    for d in sorted([root] + [p for p in root.rglob('*') if p.is_dir()]):
        level = len(d.relative_to(root).parts)
        if level > depth:
            continue
        n = sum(1 for f in d.iterdir() if f.is_file())
        print('    ' * level + f'{d.name}/  ({n} files)')
tree(RAW)

---
## 3. 형식 확인과 변환 → `data.yaml`

공개 데이터셋은 라벨 형식이 제각각입니다. 아래 셀이 **자동으로 형식을 알아내** YOLO 형식으로 바꿉니다.

| 형식 | 생김새 | 처리 |
|---|---|---|
| YOLO | 이미지마다 같은 이름의 `.txt`, 한 줄 = `클래스 cx cy w h` (0~1) | 그대로 사용 |
| Pascal VOC | 이미지마다 같은 이름의 `.xml`, 픽셀 좌표 `xmin ymin xmax ymax` | YOLO로 변환 |
| COCO | `.json` 하나에 모든 이미지와 박스, `bbox = [x, y, w, h]` 픽셀 | YOLO로 변환 |

원본에 `train`/`val`(`valid`)/`test` 폴더가 있으면 그 구분을 따르고, 없으면 7:2:1로 무작위 분할합니다(`SEED` 고정).
YOLO 형식인데 클래스 이름 파일(`data.yaml`, `classes.txt`, `obj.names`)이 없으면 `CLASS_NAMES`를 직접 채워야 합니다.

In [ ]:
CLASS_NAMES = None        # ✏️ YOLO 형식인데 클래스 이름 파일이 없을 때만: ['helmet', 'head', ...]
SPLIT_RATIO = (0.7, 0.2, 0.1)
SEED = 0

def detect_format(root):
    xmls = list(root.rglob('*.xml'))
    jsons = [j for j in root.rglob('*.json') if j.stat().st_size > 200]
    coco = []
    for j in jsons:
        try:
            d = json.load(open(j))
            if isinstance(d, dict) and 'annotations' in d and 'images' in d:
                coco.append(j)
        except Exception:
            pass
    txts = [t for t in root.rglob('*.txt') if t.name not in ('classes.txt', 'README.txt', 'readme.txt')]
    if len(xmls) > 10:
        return 'voc', xmls
    if coco:
        return 'coco', coco
    if len(txts) > 10:
        return 'yolo', txts
    return 'unknown', []

def split_of(path):
    """경로에 들어 있는 train/val/test 폴더 이름 → 분할 이름 (없으면 None)"""
    parts = [p.lower() for p in Path(path).parts]
    for name, keys in [('train', ('train', 'training')), ('val', ('val', 'valid', 'validation')), ('test', ('test', 'testing'))]:
        if any(k in parts for k in keys):
            return name
    return None

FMT, LABEL_FILES = detect_format(RAW)
images = {p.stem: p for p in RAW.rglob('*') if p.suffix.lower() in IMG_EXT}
print('라벨 형식:', FMT, '| 라벨 파일', len(LABEL_FILES), '개 | 이미지', len(images), '장')

In [ ]:
def voc_records(xmls):
    recs, names = [], []
    for x in xmls:
        r = ET.parse(x).getroot()
        fn = r.findtext('filename') or x.stem
        stem = Path(fn).stem if Path(fn).stem in images else x.stem
        size = r.find('size')
        w, h = float(size.findtext('width') or 0), float(size.findtext('height') or 0)
        if (w == 0 or h == 0) and stem in images:
            h, w = cv2.imread(str(images[stem])).shape[:2]
        boxes = []
        for o in r.findall('object'):
            n = o.findtext('name').strip()
            if n not in names:
                names.append(n)
            bb = o.find('bndbox')
            x1, y1, x2, y2 = (float(bb.findtext(k)) for k in ('xmin', 'ymin', 'xmax', 'ymax'))
            boxes.append((names.index(n), (x1 + x2) / 2 / w, (y1 + y2) / 2 / h, (x2 - x1) / w, (y2 - y1) / h))
        recs.append((stem, boxes, split_of(x)))
    return recs, names

def coco_records(jsons):
    recs, names, cat_index = [], [], {}
    for j in jsons:
        d = json.load(open(j))
        for c in d['categories']:
            if c['name'] not in names:
                names.append(c['name'])
            cat_index[(j, c['id'])] = names.index(c['name'])
        by_img = defaultdict(list)
        for a in d['annotations']:
            by_img[a['image_id']].append(a)
        for im in d['images']:
            w, h = im['width'], im['height']
            boxes = [(cat_index[(j, a['category_id'])], (a['bbox'][0] + a['bbox'][2] / 2) / w,
                      (a['bbox'][1] + a['bbox'][3] / 2) / h, a['bbox'][2] / w, a['bbox'][3] / h)
                     for a in by_img[im['id']]]
            recs.append((Path(im['file_name']).stem, boxes, split_of(j)))
    return recs, names

def yolo_records(txts):
    recs = []
    for t in txts:
        if t.stem not in images:
            continue
        boxes = []
        for line in t.read_text().strip().splitlines():
            v = line.split()
            if len(v) == 5:
                boxes.append((int(v[0]), *map(float, v[1:])))
            elif len(v) > 5 and len(v) % 2 == 1:          # 다각형 → 감싸는 박스
                xs, ys = list(map(float, v[1::2])), list(map(float, v[2::2]))
                boxes.append((int(v[0]), (min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2, max(xs) - min(xs), max(ys) - min(ys)))
        recs.append((t.stem, boxes, split_of(t)))
    names = CLASS_NAMES
    if names is None:
        for cand in list(RAW.rglob('*.yaml')) + list(RAW.rglob('classes.txt')) + list(RAW.rglob('*.names')):
            if cand.suffix == '.yaml':
                import yaml
                y = yaml.safe_load(open(cand))
                if isinstance(y, dict) and 'names' in y:
                    n = y['names']
                    names = [n[k] for k in sorted(n)] if isinstance(n, dict) else list(n)
                    break
            else:
                names = [l.strip() for l in cand.read_text().splitlines() if l.strip()]
                break
    assert names, '클래스 이름을 찾지 못했습니다. 위 셀의 CLASS_NAMES를 채우세요'
    return recs, names

if ULTRA_DATA is not None and FMT == 'yolo' and CLASS_NAMES is None:
    CLASS_NAMES = [ULTRA_DATA['names'][k] for k in sorted(ULTRA_DATA['names'])]
RECS, NAMES = {'voc': voc_records, 'coco': coco_records, 'yolo': yolo_records}[FMT](LABEL_FILES)
print('라벨 묶음', len(RECS), '개 | 클래스', len(NAMES), '개:', NAMES)

In [ ]:
def build_dataset(recs, names):
    if DATA.exists():
        shutil.rmtree(DATA)
    rng = random.Random(SEED)
    has_split = sum(r[2] is not None for r in recs) > 0.9 * len(recs)
    counts = Counter()
    for stem, boxes, sp in recs:
        if stem not in images:
            continue
        if not has_split or sp is None:
            u = rng.random()
            sp = 'train' if u < SPLIT_RATIO[0] else 'val' if u < SPLIT_RATIO[0] + SPLIT_RATIO[1] else 'test'
        src = images[stem]
        (DATA / 'images' / sp).mkdir(parents=True, exist_ok=True)
        (DATA / 'labels' / sp).mkdir(parents=True, exist_ok=True)
        shutil.copy(src, DATA / 'images' / sp / src.name)
        lines = [f'{c} {min(max(cx, 0), 1):.6f} {min(max(cy, 0), 1):.6f} {min(w, 1):.6f} {min(h, 1):.6f}'
                 for c, cx, cy, w, h in boxes if w > 0 and h > 0]
        (DATA / 'labels' / sp / (src.stem + '.txt')).write_text('\n'.join(lines) + ('\n' if lines else ''))
        counts[sp] += 1
    if counts['val'] == 0:                     # 원본에 val이 없으면 train 일부를 val로
        tr = sorted((DATA / 'images' / 'train').iterdir())
        rng.shuffle(tr)
        (DATA / 'images' / 'val').mkdir(parents=True, exist_ok=True); (DATA / 'labels' / 'val').mkdir(parents=True, exist_ok=True)
        for p in tr[:max(1, len(tr) // 5)]:
            shutil.move(p, DATA / 'images' / 'val' / p.name)
            shutil.move(DATA / 'labels' / 'train' / (p.stem + '.txt'), DATA / 'labels' / 'val' / (p.stem + '.txt'))
            counts['train'] -= 1; counts['val'] += 1
    cfg = {'path': str(DATA.resolve()), 'train': 'images/train', 'val': 'images/val',
           'names': {i: n for i, n in enumerate(names)}}
    if counts['test']:
        cfg['test'] = 'images/test'
    (DATA / 'data.yaml').write_text(json.dumps(cfg, ensure_ascii=False, indent=1))
    return str(DATA / 'data.yaml'), dict(counts), has_split

DATA_YAML, SPLIT_COUNTS, KEPT_SPLIT = build_dataset(RECS, NAMES)
print('원본 분할 유지:' if KEPT_SPLIT else '무작위 7:2:1 분할:', SPLIT_COUNTS)
print('데이터 설정 파일:', DATA_YAML)
print(open(DATA_YAML).read())

---
## 4. 데이터 살펴보기

학습 전에 데이터를 눈으로 확인합니다. 여기서 발견한 문제(클래스 불균형, 아주 작은 물체, 잘못된 라벨)가 뒤의 성능을 대부분 설명합니다.

In [ ]:
rows = []
for sp in ['train', 'val', 'test']:
    for f in (DATA / 'labels' / sp).glob('*.txt') if (DATA / 'labels' / sp).exists() else []:
        for line in f.read_text().strip().splitlines():
            c, cx, cy, w, h = line.split()
            rows.append({'split': sp, 'class': NAMES[int(c)], 'w': float(w), 'h': float(h)})
boxes_df = pd.DataFrame(rows)
boxes_df['area_pct'] = boxes_df['w'] * boxes_df['h'] * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
boxes_df[boxes_df['split'] == 'train']['class'].value_counts().plot.bar(ax=axes[0], color='tab:blue')
axes[0].set_title('Boxes per class (train)'); axes[0].set_ylabel('boxes')
axes[1].hist(boxes_df['area_pct'], bins=40, color='tab:gray', log=True)
axes[1].set_xlabel('box area (% of image)'); axes[1].set_title('Box size distribution')
plt.tight_layout(); plt.savefig('eda.png', dpi=110); save_out('eda.png'); plt.show()
print('이미지 1% 미만 크기의 작은 박스 비율:', round((boxes_df['area_pct'] < 1).mean(), 3))

In [ ]:
# 정답 라벨을 그린 샘플 6장
palette = [tuple(int(c) for c in np.random.default_rng(i).integers(60, 255, 3)) for i in range(len(NAMES))]
samples = sorted((DATA / 'images' / 'train').iterdir())
random.Random(SEED).shuffle(samples)
imgs, titles = [], []
for p in samples[:6]:
    im = cv2.imread(str(p)); h, w = im.shape[:2]
    lbl = (DATA / 'labels' / 'train' / (p.stem + '.txt')).read_text().strip().splitlines()
    for line in lbl:
        c, cx, cy, bw, bh = line.split(); c = int(c)
        x1, y1 = int((float(cx) - float(bw) / 2) * w), int((float(cy) - float(bh) / 2) * h)
        x2, y2 = int((float(cx) + float(bw) / 2) * w), int((float(cy) + float(bh) / 2) * h)
        cv2.rectangle(im, (x1, y1), (x2, y2), palette[c], 2)
        cv2.putText(im, NAMES[c], (x1, max(12, y1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, palette[c], 2)
    imgs.append(im); titles.append(f'{p.name[:25]} | {len(lbl)} boxes')
show_grid(imgs, titles)

**확인해 보기** ✏️ — 아래 질문의 답을 이 셀 아래에 메모해 두세요. 멘토링 때 함께 봅니다.
- 클래스마다 박스 수가 몇 배나 차이 나나요? 적은 클래스는 성능이 낮게 나올 가능성이 큽니다.
- 박스가 아주 작은 이미지가 많나요? 그렇다면 `imgsz`를 키우는 실험이 필요합니다.
- 샘플 그림에서 빠진 라벨이나 이상한 박스가 보이나요?

---
## 5. 베이스라인 학습

먼저 **작은 모델, 기본 설정**으로 끝까지 한 번 돌려 기준점(베이스라인)을 만듭니다. 개선은 그다음입니다.
`QUICK_RUN = True`로 두면 5 epoch만 돌려 파이프라인이 끝까지 도는지 빠르게 확인할 수 있습니다.
데이터가 커서 1 epoch가 10분 넘게 걸리면 `FRACTION`을 0.3처럼 낮춰 학습 이미지의 일부만 씁니다.

In [ ]:
QUICK_RUN = True                              # ✏️ 파이프라인 확인 후 False로 바꿔 본 학습
BASE_MODEL = 'yolo26n.pt' if TASK == 'detect' else 'yolo26n-seg.pt'
FRACTION = 1.0                                # ✏️ 학습 이미지 중 사용할 비율 (큰 데이터는 0.3 등)
CFG = dict(epochs=5, imgsz=640, batch=16, fraction=FRACTION) if QUICK_RUN else dict(epochs=50, imgsz=640, batch=16, fraction=FRACTION)

def train(name, model=BASE_MODEL, **kw):
    cfg = {**CFG, **kw}
    m = YOLO(model)
    t0 = time.time()
    m.train(data=DATA_YAML, project='runs_ws', name=name, exist_ok=True, seed=SEED, verbose=False, **cfg)
    best = Path(m.trainer.save_dir) / 'weights' / 'best.pt'
    shutil.copy(best, SAVE_DIR / f'{name}_best.pt')
    return best, round((time.time() - t0) / 60, 1), cfg

BEST, BASE_MIN, BASE_CFG = train('baseline')
print(f'학습 {BASE_MIN}분 | 최고 가중치: {BEST} (Drive에도 저장)')
display(Image(filename=str(Path(BEST).parent.parent / 'results.png'), width=900))

---
## 6. 평가 — 지표, 임계값, 오류 분석

### 6-1. 클래스별 지표

전체 mAP 하나만 보면 안 됩니다. **우리 문제에서 중요한 클래스**의 재현율·정밀도를 따로 봐야 성공 기준(`SUCCESS`)과 비교할 수 있습니다.

In [ ]:
EVAL_SPLIT = 'test' if SPLIT_COUNTS.get('test') else 'val'
model = YOLO(BEST)
res = model.val(data=DATA_YAML, split=EVAL_SPLIT, verbose=False, plots=True, project='runs_ws', name='eval', exist_ok=True)
per_class = pd.DataFrame({
    'class': [NAMES[i] for i in res.box.ap_class_index],
    'precision': np.round(res.box.p, 3), 'recall': np.round(res.box.r, 3),
    'ap50': np.round(res.box.ap50, 3), 'ap50_95': np.round(res.box.ap, 3)})
per_class.to_csv('per_class.csv', index=False); save_out('per_class.csv')
print(f'{EVAL_SPLIT} 세트 | mAP50 {res.box.map50:.3f} | mAP50-95 {res.box.map:.3f}')
per_class

In [ ]:
EVAL_DIR = Path(res.save_dir)
for name in ['confusion_matrix_normalized.png', 'BoxPR_curve.png']:
    if (EVAL_DIR / name).exists():
        display(Image(filename=str(EVAL_DIR / name), width=520))

### 6-2. 신뢰도 임계값 정하기

서비스에서는 임계값 하나를 골라야 합니다. 평가 세트 예측을 한 번만 해 두고, 임계값을 바꿔 가며 **클래스를 구분해** TP·FP·FN을 셉니다. (실습 A Step 2에서 만든 방법과 같습니다.)

In [ ]:
def iou_mat(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); ab = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (aa[:, None] + ab[None, :] - inter + 1e-9)

def gt_boxes(img_path, split):
    h, w = cv2.imread(str(img_path)).shape[:2]
    out = []
    for line in (DATA / 'labels' / split / (img_path.stem + '.txt')).read_text().strip().splitlines():
        c, cx, cy, bw, bh = map(float, line.split())
        out.append([(cx - bw / 2) * w, (cy - bh / 2) * h, (cx + bw / 2) * w, (cy + bh / 2) * h, c])
    return np.array(out).reshape(-1, 5)

EVAL_IMAGES = sorted((DATA / 'images' / EVAL_SPLIT).iterdir())
cache = []
for p in EVAL_IMAGES:
    r = model(str(p), conf=0.05, verbose=False)[0]
    pred = np.hstack([r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()[:, None], r.boxes.cls.cpu().numpy()[:, None]])
    cache.append((p, pred, gt_boxes(p, EVAL_SPLIT)))

def count(thr, cls=None):
    tp = fp = fn = 0
    for _, pred, gt in cache:
        for c in (range(len(NAMES)) if cls is None else [cls]):
            P = pred[(pred[:, 4] >= thr) & (pred[:, 5] == c)]; G = gt[gt[:, 4] == c]
            ious, used, hit = iou_mat(P[:, :4], G[:, :4]), set(), 0
            for i in np.argsort(-P[:, 4]):
                if len(G) == 0:
                    break
                j = int(np.argmax(np.where([k in used for k in range(len(G))], -1, ious[i])))
                if ious[i, j] >= 0.5 and j not in used:
                    used.add(j); hit += 1
            tp += hit; fp += len(P) - hit; fn += len(G) - hit
    prec, rec = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    return {'conf': thr, 'tp': tp, 'fp': fp, 'fn': fn, 'precision': round(prec, 3), 'recall': round(rec, 3),
            'f1': round(2 * prec * rec / max(prec + rec, 1e-9), 3)}

thr_table = pd.DataFrame([count(t) for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]])
BEST_CONF = float(thr_table.loc[thr_table['f1'].idxmax(), 'conf'])
print('전체 F1이 가장 높은 임계값:', BEST_CONF)
thr_table

In [ ]:
FOCUS_CLASS = NAMES[0]          # ✏️ 우리 문제에서 가장 중요한 클래스 이름 (예: 'no_helmet')
fc = NAMES.index(FOCUS_CLASS)
focus = pd.DataFrame([count(t, fc) for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]])
print(f'"{FOCUS_CLASS}" 클래스만 따로 본 임계값별 성능')
focus

**생각해 보기** — 우리 문제에서 놓침(FN)과 헛경보(FP) 중 무엇이 더 비싼가요? 놓침이 위험하면 재현율이 목표를 넘는 가장 높은 임계값을, 헛경보가 비싸면 정밀도가 목표를 넘는 가장 낮은 임계값을 고릅니다. 고른 값을 아래 `SERVICE_CONF`에 적으세요.

In [ ]:
SERVICE_CONF = BEST_CONF        # ✏️ 위 표를 보고 우리 문제에 맞는 값으로

### 6-3. 오류 분석 — 틀린 사진 직접 보기

In [ ]:
errs = []
for p, pred, gt in cache:
    P = pred[pred[:, 4] >= SERVICE_CONF]
    errs.append((abs(len(P) - len(gt)) + len(gt) - min(len(P), len(gt)), p, P, gt))
errs.sort(key=lambda e: -e[0])

imgs, titles = [], []
for e, p, P, gt in errs[:6]:
    im = cv2.imread(str(p))
    for x1, y1, x2, y2, c in gt:
        cv2.rectangle(im, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 3)
    for x1, y1, x2, y2, cf, c in P:
        cv2.rectangle(im, (int(x1), int(y1)), (int(x2), int(y2)), (0, 0, 255), 2)
        cv2.putText(im, f'{NAMES[int(c)]} {cf:.2f}', (int(x1), max(12, int(y1) - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 255), 1)
    imgs.append(im); titles.append(f'GT {len(gt)} (green) / pred {len(P)} (red) | {p.name[:20]}')
show_grid(imgs, titles)

**오류 분류표** ✏️ — 위 사진들을 보고 원인별로 몇 장인지 세어 적으세요. 7단계 개선 실험의 방향이 여기서 나옵니다.

| 원인 | 장 수 | 개선 아이디어 |
|---|---|---|
| 작은 물체를 놓침 | | `imgsz` 키우기 |
| 비슷한 물체와 헷갈림 | | 더 큰 모델, 헷갈리는 사례 데이터 추가 |
| 가려짐·겹침 | | 데이터 증강, 데이터 추가 |
| 라벨 자체가 틀림 | | 라벨 수정 |

---
## 7. 개선 실험 기록

한 번에 **하나만** 바꿉니다. 실험마다 이름, 바꾼 것, 이유를 적고, 결과는 표에 자동으로 쌓입니다.

In [ ]:
EXPERIMENTS = [
    # ✏️ 이름, 바꿀 설정, 이유 — 오류 분류표를 근거로 1~3개
    {'name': 'imgsz_960', 'change': {'imgsz': 960}, 'why': '작은 물체 놓침이 많아서'},
    # {'name': 'model_s', 'change': {'model': 'yolo26s.pt'}, 'why': '비슷한 물체 혼동'},
    # {'name': 'more_epochs', 'change': {'epochs': 100}, 'why': '손실이 아직 내려가는 중'},
]
RUN_EXPERIMENTS = False         # ✏️ True로 바꾸면 위 실험을 차례로 학습

def evaluate(weights):
    r = YOLO(weights).val(data=DATA_YAML, split=EVAL_SPLIT, verbose=False, plots=False)
    ap = dict(zip(r.box.ap_class_index.tolist(), r.box.ap50.tolist()))
    return {'map50': round(r.box.map50, 3), 'map50_95': round(r.box.map, 3), f'ap50_{FOCUS_CLASS}': round(ap.get(fc, 0), 3)}

log_path = SAVE_DIR / 'experiments.csv'
log = pd.read_csv(log_path).to_dict('records') if log_path.exists() else []
if not any(r['name'] == 'baseline' for r in log):
    log.append({'name': 'baseline', 'change': '-', 'why': '기준점', 'train_min': BASE_MIN, **evaluate(BEST)})
if RUN_EXPERIMENTS:
    for e in EXPERIMENTS:
        if any(r['name'] == e['name'] for r in log):
            continue
        ch = dict(e['change']); mdl = ch.pop('model', BASE_MODEL)
        w, mins, _ = train(e['name'], model=mdl, **ch)
        log.append({'name': e['name'], 'change': json.dumps(e['change']), 'why': e['why'], 'train_min': mins, **evaluate(w)})
exp = pd.DataFrame(log)
exp.to_csv(log_path, index=False)
exp

실험 기록은 Drive의 `experiments.csv`에 계속 쌓이므로, 런타임이 끊기거나 다음 시간에 이어서 해도 기록이 남습니다. 가장 좋은 실험의 가중치를 다음 셀에서 고릅니다.

In [ ]:
best_row = exp.loc[exp['map50'].idxmax()]
FINAL_WEIGHTS = SAVE_DIR / f"{best_row['name']}_best.pt"
print('최종 모델:', best_row['name'], '| mAP50', best_row['map50'], '|', FINAL_WEIGHTS)

---
## 8. 데모 — 새 이미지·영상에 적용

평가 세트에서 몇 장을 골라 최종 모델로 예측하고, 원하면 **직접 가져온 사진이나 영상**도 올려 봅니다.

In [ ]:
final = YOLO(FINAL_WEIGHTS)
demo_imgs = [final(str(p), conf=SERVICE_CONF, verbose=False)[0].plot() for p in EVAL_IMAGES[:6]]
show_grid(demo_imgs, [p.name[:30] for p in EVAL_IMAGES[:6]])
for i, im in enumerate(demo_imgs[:3]):
    cv2.imwrite(f'demo_{i + 1}.jpg', im); save_out(f'demo_{i + 1}.jpg')

In [ ]:
UPLOAD_DEMO = False             # ✏️ True로 바꾸면 사진·영상 파일을 올려 예측
if UPLOAD_DEMO and IN_COLAB:
    up = files.upload()
    for name in up:
        ext = Path(name).suffix.lower()
        safe = f'upload_{len(list(Path(".").glob("upload_*"))) + 1}{ext}'
        os.replace(name, safe)
        res_u = final.predict(safe, conf=SERVICE_CONF, save=True, project='runs_ws', name='demo', exist_ok=True, verbose=False)
        print('저장:', Path(res_u[0].save_dir))

---
## 9. 결과 정리 — 요약 리포트

프로젝트 카드, 데이터 요약, 실험 기록, 클래스별 성능, 성공 기준 달성 여부를 한 파일(`project_report.md`)로 자동 정리합니다. 결과 공유나 멘토링 때 이 파일과 `report.png`를 보여 주면 됩니다.

In [ ]:
focus_at = count(SERVICE_CONF, fc)
lines = [
    f'# {PROJECT_NAME}', '',
    f'- Team: {TEAM}', f'- Problem: {PROBLEM}', f'- Success criterion: {SUCCESS}', '',
    '## Data',
    f'- Source: {DATA_SOURCE} ({ {"kaggle": KAGGLE_SLUG, "url": ZIP_URL, "drive": DRIVE_ZIP, "ultralytics": ULTRA_YAML}[DATA_SOURCE] })',
    f'- Label format: {FMT} | classes ({len(NAMES)}): {", ".join(NAMES)}',
    f'- Split: {SPLIT_COUNTS} ({"original split" if KEPT_SPLIT else "random 7:2:1"})', '',
    '## Experiments', exp.to_markdown(index=False), '',
    f'## Per-class performance ({EVAL_SPLIT}, baseline)', per_class.to_markdown(index=False), '',
    f'## Service threshold: conf {SERVICE_CONF}',
    f'- {FOCUS_CLASS}: precision {focus_at["precision"]}, recall {focus_at["recall"]} (TP {focus_at["tp"]}, FP {focus_at["fp"]}, FN {focus_at["fn"]})', '',
    '## Notes (fill in)', '- What worked:', '- What did not work:', '- Next step:',
]
Path('project_report.md').write_text('\n'.join(lines), encoding='utf-8'); save_out('project_report.md')

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].bar(exp['name'], exp['map50'], color='tab:blue'); axes[0].set_ylim(0, 1)
axes[0].set_title('mAP50 by experiment'); axes[0].tick_params(axis='x', rotation=20)
axes[1].plot(focus['conf'], focus['precision'], '-o', label='precision')
axes[1].plot(focus['conf'], focus['recall'], '-o', label='recall')
axes[1].axvline(SERVICE_CONF, ls='--', c='gray'); axes[1].set_ylim(0, 1.05)
axes[1].set_title(f'{FOCUS_CLASS}: threshold trade-off'); axes[1].legend(); axes[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig('report.png', dpi=120); save_out('report.png'); plt.show()
print(open('project_report.md', encoding='utf-8').read())
print('\nDrive 저장 파일:', sorted(p.name for p in SAVE_DIR.iterdir()))

---
## 마무리 체크리스트

- [ ] 프로젝트 카드의 성공 기준과 실제 결과를 비교해 `project_report.md`의 Notes를 채웠다
- [ ] 실험을 최소 1개 이상 해서 베이스라인과 비교했다
- [ ] 오류 분류표를 채우고, 개선 실험이 그 근거와 연결된다
- [ ] Drive에 `*_best.pt`, `experiments.csv`, `project_report.md`, `report.png`가 있다